# LunarLander overestimation

DDPG and TD3 with no normalization, critic LayerNorm, and actor plus critic LayerNorm.


In [ ]:
from pathlib import Path

import torch

from IPython.display import Image, display

from lunar_overconfidence.analysis import (
    analyze_bias_performance,
    save_bias_performance,
)
from lunar_overconfidence.experiments import (
    create_experiment,
    evaluate_experiment,
    load_experiment,
    plot_diagnostics,
    plot_experiment,
    train_experiment,
)


## Settings


In [ ]:
experiment_name = "layer-norm-study-2"
seeds = [1, 2, 3, 4, 5]
log_losses = True

torch.set_num_threads(1)

normalization_variants = {
    "baseline": {"layer_norm": False, "actor_layer_norm": False},
    "layer-norm": {"layer_norm": True, "actor_layer_norm": False},
    "actor-critic-layer-norm": {"layer_norm": True, "actor_layer_norm": True},
}

training = {
    "env_name": "LunarLanderContinuous-v3",
    "max_steps": 300_000,
    "n_envs": 1,
    "steps_per_update": 1,
    "learning_starts": 10_000,
    "buffer_size": 200_000,
    "batch_size": 256,
    "gamma": 0.98,
    "tau": 0.005,
    "action_noise": 0.1,
    "actor_hidden": (256, 256),
    "critic_hidden": (256, 256),
    "lr_actor": 1e-3,
    "lr_critic": 1e-3,
    "eval_interval": 10_000,
    "n_eval_episodes": 10,
}
td3_settings = {"policy_delay": 2, "target_noise": 0.2, "target_noise_clip": 0.5}
evaluation = {"n_episodes": 100, "seed": 10_000, "max_steps": 10_000}

comparisons = {
    "critic-vs-none": ("baseline", "layer-norm"),
    "both-vs-none": ("baseline", "actor-critic-layer-norm"),
    "both-vs-critic": ("layer-norm", "actor-critic-layer-norm"),
}
comparison_step = None
interval = "seeds"


## Reopen saved results

Loads saved evaluations without training or evaluating again.


In [ ]:
saved_experiment = None

if saved_experiment is not None:
    experiment_dir = Path(saved_experiment)
    results = load_experiment(experiment_dir)


## Create an experiment

Creates a timestamped folder and saves the configuration.


In [ ]:
experiment_dir = create_experiment(
    experiment_name, training, seeds, evaluation, td3_settings,
    normalization_variants=normalization_variants,
)
experiment_dir


## Train


In [ ]:
models = train_experiment(experiment_dir, log_losses=log_losses)


## Evaluate

Compares critic predictions with discounted returns from held-out trajectories. Trajectories that reach the cap are excluded from bias estimates.


In [ ]:
results = evaluate_experiment(experiment_dir)


## Analyze

Compares bias and reward at a shared checkpoint. Pairwise differences are the second condition minus the first.


In [ ]:
reports = {}
for name, (baseline, normalized) in comparisons.items():
    report = analyze_bias_performance(
        results, step=comparison_step, baseline=baseline, normalized=normalized
    )
    reports[name] = report
    save_bias_performance(report, experiment_dir / "tables" / name)
    print(name)
    display(report["normalization_pairs"])
    display(report["notes"])


## Main figures

Thin curves show individual training seeds; thick curves show their means.


In [ ]:
figures = {}
for name, (baseline, normalized) in comparisons.items():
    figures[name] = plot_experiment(
        results, reports[name], experiment_dir / name, baseline, normalized, interval
    )
    print(name)
    display(Image(filename=str(figures[name]["learning_curves"]["png"])))


In [ ]:
for name in comparisons:
    print(name)
    display(Image(filename=str(figures[name]["bias_performance"]["png"])))


In [ ]:
for name in comparisons:
    print(name)
    display(Image(filename=str(figures[name]["normalization_effects"]["png"])))


## Diagnostics


In [ ]:
diagnostics = {}
for name, (baseline, normalized) in comparisons.items():
    diagnostics[name] = plot_diagnostics(
        results, experiment_dir / name, baseline, normalized
    )
    print(name)
    display(Image(filename=str(diagnostics[name]["completion_rates"]["png"])))
    if diagnostics[name]["td3_critics"] is not None:
        display(Image(filename=str(diagnostics[name]["td3_critics"]["png"])))


## TensorBoard and video

TensorBoard uses training logs; video uses a model returned by the training cell.


In [ ]:
show_tensorboard = True
if show_tensorboard:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", f"--logdir {experiment_dir}")


In [ ]:
show_video = True
if show_video:
    models[("td3", "layer-norm", seeds[0])].visualize_best(video=True)
